In [58]:
import pandas as pd
import random
import dgl
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from dgllife.model import model_zoo
from dgllife.utils import smiles_to_bigraph
from dgllife.utils import EarlyStopping, Meter
from dgllife.utils import AttentiveFPAtomFeaturizer
from dgllife.utils import AttentiveFPBondFeaturizer

import torch
import os
import random
import numpy as np
import ast

import matplotlib
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import pandas as pd
from rdkit.Chem import AllChem
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
from IPython.display import SVG, display
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import pickle
import argparse
from rdkit import RDLogger 
import warnings
warnings.filterwarnings("ignore")
RDLogger.DisableLog('rdApp.*') # switch off RDKit warning messages

In [59]:
from utils import get_values_at_positions, atom_finder, smiles_augmentation, concat_feature_reactive_atom, collate_molgraphs, Canon_SMILES_similarity
from model import AttentiveFPPredictor_rxn, weighted_binary_cross_entropy

In [60]:
#Assign device 
device = "cpu"

In [61]:
atom_featurizer = AttentiveFPAtomFeaturizer(atom_data_field='hv')
bond_featurizer = AttentiveFPBondFeaturizer(bond_data_field='he')
n_feats = atom_featurizer.feat_size('hv')
e_feats = bond_featurizer.feat_size('he')
print( 'Number of features in graph : ' , n_feats)

Number of features in graph :  39


In [62]:
# Laoding the trained model to fit your classification task
fn = 'trained_classifier_wo_augm_10_07_25_WCE'
model = AttentiveFPPredictor_rxn(node_feat_size=n_feats,
                                   edge_feat_size=e_feats,
                                   num_layers=2,
                                   num_timesteps=1,
                                   graph_feat_size=200,
                                   n_tasks=1,
                                   dropout=0.1
                                    )
model.load_state_dict(torch.load(fn,map_location=device))

<All keys matched successfully>

In [63]:
def classifier_elementary_reactive(trained_model, sm, node_featurizer, edge_featurizer):
    smiles_graph = smiles_to_bigraph(sm, node_featurizer=node_featurizer,edge_featurizer=edge_featurizer, canonical_atom_order=False)
    n_feats_sm = smiles_graph.ndata.pop('hv').to(device)
    e_feats_sm = smiles_graph.edata.pop('he').to(device)
    pred_val, graph_feat = model(smiles_graph, n_feats_sm, e_feats_sm)
    pred_val_np = pred_val.detach().cpu().numpy()[0]
    threshold_class = 0.5
    pred_val_bin = [1 if pred_val_np >= threshold_class else 0]
    pred_val_bin
    return pred_val_bin


In [79]:
smiles_string = 'COc1c(c2c(C(C)C)cc(C(C)C)cc2C(C)C)c([P+](C(C)(C)C)(C(C)(C)C)[Pd](c3ccc(C(F)(F)F)cc3)Br)c(OC)cc1.CC(C)([O-])C.OCCC4CCCCC4.[Na+]'

In [80]:
bin_pred = classifier_elementary_reactive(model, smiles_string, atom_featurizer, bond_featurizer)

In [81]:
bin_pred

[1]